# 04 — sparkMeasure Metrics Explained

How to read sparkMeasure stage metrics and translate them into Spark performance decisions.

## 00 — Spark setup

Standard SparkSession setup used across the local Spark cluster notebooks.

In [1]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd

spark = (
    SparkSession.builder
    .appName("sparkmeasure_metrics_explained")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.debug.maxToStringFields", "1000")

spark

## 01 — Example sparkMeasure report

This is the stage metrics report that we will interpret step by step.

In [2]:
metrics = {
    "numStages": 4,
    "numTasks": 9,
    "elapsedTime_ms": 700,
    "stageDuration_ms": 713,
    "executorRunTime_ms": 938,
    "executorCpuTime_ms": 925,
    "executorDeserializeTime_ms": 48,
    "executorDeserializeCpuTime_ms": 16,
    "resultSerializationTime_ms": 0,
    "jvmGCTime_ms": 10,
    "shuffleFetchWaitTime_ms": 0,
    "shuffleWriteTime_ms": 7,
    "resultSize_bytes": 12524,
    "diskBytesSpilled": 0,
    "memoryBytesSpilled": 0,
    "peakExecutionMemory_bytes": 574684976,
    "recordsRead": 2200000,
    "bytesRead": 0,
    "recordsWritten": 0,
    "bytesWritten": 0,
    "shuffleRecordsRead": 2200002,
    "shuffleTotalBlocksFetched": 12,
    "shuffleLocalBlocksFetched": 12,
    "shuffleRemoteBlocksFetched": 0,
    "shuffleTotalBytesRead": 4029442,
    "shuffleLocalBytesRead": 4029442,
    "shuffleRemoteBytesRead": 0,
    "shuffleRemoteBytesReadToDisk": 0,
    "shuffleBytesWritten": 4029442,
    "shuffleRecordsWritten": 2200002,
}

pd.DataFrame(metrics.items(), columns=["metric", "value"])

,metric,value
0,numStages,4
1,numTasks,9
2,elapsedTime_ms,700
3,stageDuration_ms,713
4,executorRunTime_ms,938
5,executorCpuTime_ms,925
6,executorDeserializeTime_ms,48
7,executorDeserializeCpuTime_ms,16
8,resultSerializationTime_ms,0
9,jvmGCTime_ms,10


## 02 — High-level interpretation

Read the report as a story: stages, tasks, wall time, executor time, CPU, shuffle, spill, GC, and result size.

In [3]:
summary = [
    ("numStages", "4 stages", "The job had several execution phases. Normal for joins/aggregations."),
    ("numTasks", "9 tasks", "Small number of tasks. Not a highly parallel workload."),
    ("elapsedTime", "0.7 s", "Wall-clock runtime from the driver perspective."),
    ("executorRunTime", "0.9 s", "Total task runtime; can exceed elapsed time because tasks run in parallel."),
    ("executorCpuTime", "0.9 s", "Almost equal to executor runtime, so tasks were CPU-active."),
    ("shuffleTotalBytesRead", "3.8 MB", "Shuffle exists, but it is small."),
    ("shuffleRemoteBytesRead", "0 Bytes", "No remote shuffle network bottleneck."),
    ("spill", "0 Bytes", "Memory was sufficient."),
    ("jvmGCTime", "10 ms", "GC overhead is negligible."),
]
pd.DataFrame(summary, columns=["metric", "observed", "interpretation"])

,metric,observed,interpretation
0,numStages,4 stages,The job had several execution phases. Normal f...
1,numTasks,9 tasks,Small number of tasks. Not a highly parallel w...
2,elapsedTime,0.7 s,Wall-clock runtime from the driver perspective.
3,executorRunTime,0.9 s,Total task runtime; can exceed elapsed time be...
4,executorCpuTime,0.9 s,"Almost equal to executor runtime, so tasks wer..."
5,shuffleTotalBytesRead,3.8 MB,"Shuffle exists, but it is small."
6,shuffleRemoteBytesRead,0 Bytes,No remote shuffle network bottleneck.
7,spill,0 Bytes,Memory was sufficient.
8,jvmGCTime,10 ms,GC overhead is negligible.


## 03 — Stages and tasks

`numStages` and `numTasks` tell you how Spark decomposed the job.

In [4]:
pd.DataFrame([
    ("numStages", "4", "Multiple stages usually mean shuffle boundaries or separate execution phases."),
    ("numTasks", "9", "Low task count means limited parallelism. Fine for small data, suspicious for large data."),
    ("Too few tasks", "Risk", "May underuse the cluster."),
    ("Too many tiny tasks", "Risk", "May waste time on scheduling overhead."),
], columns=["item", "value", "meaning"])

,item,value,meaning
0,numStages,4,Multiple stages usually mean shuffle boundarie...
1,numTasks,9,Low task count means limited parallelism. Fine...
2,Too few tasks,Risk,May underuse the cluster.
3,Too many tiny tasks,Risk,May waste time on scheduling overhead.


## 04 — Time metrics

Time metrics are the first layer: elapsed time, stage duration, executor runtime, and CPU time.

In [5]:
pd.DataFrame([
    ("elapsedTime", "700 ms", "Wall-clock time. What the user feels."),
    ("stageDuration", "713 ms", "Sum of Spark stage durations. Close to elapsed time here."),
    ("executorRunTime", "938 ms", "Total task runtime across executors."),
    ("executorCpuTime", "925 ms", "Actual CPU time."),
    ("Conclusion", "CPU-active", "executorCpuTime is very close to executorRunTime."),
], columns=["metric", "value", "meaning"])

,metric,value,meaning
0,elapsedTime,700 ms,Wall-clock time. What the user feels.
1,stageDuration,713 ms,Sum of Spark stage durations. Close to elapsed...
2,executorRunTime,938 ms,Total task runtime across executors.
3,executorCpuTime,925 ms,Actual CPU time.
4,Conclusion,CPU-active,executorCpuTime is very close to executorRunTime.


## 05 — CPU efficiency ratio

A simple ratio helps distinguish CPU-bound work from waiting/IO/network overhead.

In [6]:
executor_run = metrics["executorRunTime_ms"]
executor_cpu = metrics["executorCpuTime_ms"]
cpu_efficiency = executor_cpu / executor_run if executor_run else None

pd.DataFrame(
    [("executorCpuTime / executorRunTime", round(cpu_efficiency, 3), "Near 1.0 means CPU-active; much lower means waiting, IO, GC, or scheduling overhead")],
    columns=["ratio", "value", "meaning"]
)

,ratio,value,meaning
0,executorCpuTime / executorRunTime,0.986,Near 1.0 means CPU-active; much lower means wa...


## 06 — Serialization and deserialization

Executor deserialization time is task startup overhead. High values can mean too many tiny tasks or huge task closures.

In [7]:
pd.DataFrame([
    ("executorDeserializeTime", "48 ms", "Small overhead."),
    ("executorDeserializeCpuTime", "16 ms", "Small CPU cost."),
    ("resultSerializationTime", "0 ms", "No meaningful result serialization cost."),
    ("Interpretation", "healthy", "Serialization overhead is not a problem."),
], columns=["metric", "value", "meaning"])

,metric,value,meaning
0,executorDeserializeTime,48 ms,Small overhead.
1,executorDeserializeCpuTime,16 ms,Small CPU cost.
2,resultSerializationTime,0 ms,No meaningful result serialization cost.
3,Interpretation,healthy,Serialization overhead is not a problem.


## 07 — Garbage collection

GC time shows JVM memory pressure.

In [8]:
gc_ratio = metrics["jvmGCTime_ms"] / metrics["executorRunTime_ms"]
pd.DataFrame(
    [("jvmGCTime / executorRunTime", round(gc_ratio, 4), "Very low. GC is not an issue here.")],
    columns=["ratio", "value", "meaning"]
)

,ratio,value,meaning
0,jvmGCTime / executorRunTime,0.0107,Very low. GC is not an issue here.


## 08 — Shuffle read/write

Shuffle metrics show whether Spark moved data between stages. Joins, groupBy, distinct, repartition, and windows often shuffle.

In [9]:
pd.DataFrame([
    ("shuffleBytesWritten", "3.8 MB", "Data written into shuffle files by upstream tasks."),
    ("shuffleTotalBytesRead", "3.8 MB", "Data read by downstream shuffle tasks."),
    ("shuffleRecordsWritten", "2,200,002", "Records written to shuffle."),
    ("shuffleRecordsRead", "2,200,002", "Records read from shuffle."),
    ("Interpretation", "small shuffle", "There is shuffle, but byte volume is small."),
], columns=["metric", "value", "meaning"])

,metric,value,meaning
0,shuffleBytesWritten,3.8 MB,Data written into shuffle files by upstream ta...
1,shuffleTotalBytesRead,3.8 MB,Data read by downstream shuffle tasks.
2,shuffleRecordsWritten,"2,200,002",Records written to shuffle.
3,shuffleRecordsRead,"2,200,002",Records read from shuffle.
4,Interpretation,small shuffle,"There is shuffle, but byte volume is small."


## 09 — Local vs remote shuffle

Remote shuffle is usually more expensive because it involves network transfer.

In [10]:
total = metrics["shuffleTotalBytesRead"]
local_ratio = metrics["shuffleLocalBytesRead"] / total if total else 0
remote_ratio = metrics["shuffleRemoteBytesRead"] / total if total else 0

pd.DataFrame([
    ("local shuffle read ratio", round(local_ratio, 3), "All shuffle read was local."),
    ("remote shuffle read ratio", round(remote_ratio, 3), "No remote shuffle network cost."),
    ("shuffleFetchWaitTime", "0 ms", "Tasks did not wait for shuffle blocks."),
], columns=["metric", "value", "meaning"])

,metric,value,meaning
0,local shuffle read ratio,1.0,All shuffle read was local.
1,remote shuffle read ratio,0.0,No remote shuffle network cost.
2,shuffleFetchWaitTime,0 ms,Tasks did not wait for shuffle blocks.


## 10 — Spill metrics

Spilling means Spark lacked execution memory and had to write intermediate data to memory/disk spill paths.

In [11]:
pd.DataFrame([
    ("memoryBytesSpilled", "0 Bytes", "No memory spill."),
    ("diskBytesSpilled", "0 Bytes", "No disk spill."),
    ("peakExecutionMemory", "548 MB approx", "Peak execution memory was used, but not enough to spill."),
    ("Interpretation", "healthy", "No evidence of memory pressure."),
], columns=["metric", "value", "meaning"])

,metric,value,meaning
0,memoryBytesSpilled,0 Bytes,No memory spill.
1,diskBytesSpilled,0 Bytes,No disk spill.
2,peakExecutionMemory,548 MB approx,"Peak execution memory was used, but not enough..."
3,Interpretation,healthy,No evidence of memory pressure.


## 11 — Input and output metrics

`recordsRead` and `bytesRead` describe source input. `bytesRead = 0` can be normal for generated ranges or in-memory transformations.

In [12]:
pd.DataFrame([
    ("recordsRead", "2,200,000", "Spark processed 2.2M input records."),
    ("bytesRead", "0 Bytes", "Likely generated/in-memory source rather than file scan."),
    ("recordsWritten", "0", "No file/table write occurred."),
    ("bytesWritten", "0 Bytes", "No output write occurred."),
    ("resultSize", "12.2 KB", "Small result returned to driver."),
], columns=["metric", "value", "meaning"])

,metric,value,meaning
0,recordsRead,"2,200,000",Spark processed 2.2M input records.
1,bytesRead,0 Bytes,Likely generated/in-memory source rather than ...
2,recordsWritten,0,No file/table write occurred.
3,bytesWritten,0 Bytes,No output write occurred.
4,resultSize,12.2 KB,Small result returned to driver.


## 12 — What this report says

Concrete interpretation of the provided metrics.

In [13]:
pd.DataFrame([
    ("Runtime", "Fast", "0.7 s elapsed."),
    ("Parallelism", "Small job", "Only 9 tasks."),
    ("CPU", "Healthy", "executorCpuTime is almost equal to executorRunTime."),
    ("Shuffle", "Present but small", "3.8 MB shuffle read/write."),
    ("Network", "No issue", "0 remote shuffle bytes and 0 fetch wait time."),
    ("Memory", "Healthy", "0 spill and low GC."),
    ("Bottleneck", "None obvious", "This looks like a small healthy job."),
], columns=["area", "status", "reason"])

,area,status,reason
0,Runtime,Fast,0.7 s elapsed.
1,Parallelism,Small job,Only 9 tasks.
2,CPU,Healthy,executorCpuTime is almost equal to executorRun...
3,Shuffle,Present but small,3.8 MB shuffle read/write.
4,Network,No issue,0 remote shuffle bytes and 0 fetch wait time.
5,Memory,Healthy,0 spill and low GC.
6,Bottleneck,None obvious,This looks like a small healthy job.


## 13 — What would look bad

Use this table when reading future sparkMeasure reports.

In [14]:
pd.DataFrame([
    ("High shuffleRemoteBytesRead", "Network-heavy shuffle", "Try broadcast, partitioning, AQE, or reduce shuffle width."),
    ("High shuffleFetchWaitTime", "Tasks waiting for shuffle data", "Check network, skew, remote blocks, executor pressure."),
    ("High diskBytesSpilled", "Memory pressure", "Increase memory, reduce partitions, pre-aggregate, avoid wide rows."),
    ("High jvmGCTime", "GC pressure", "Tune memory, reduce caching, avoid huge objects/UDFs."),
    ("CPU time much lower than run time", "Waiting/IO/scheduling overhead", "Look at shuffle wait, GC, spill, input scan, task overhead."),
    ("Very high max task time vs median", "Skew", "Use partition metrics, AQE skew handling, salting."),
    ("Huge resultSize", "Driver pressure", "Avoid collect/show on large results."),
], columns=["symptom", "likely_problem", "typical_fix"])

,symptom,likely_problem,typical_fix
0,High shuffleRemoteBytesRead,Network-heavy shuffle,"Try broadcast, partitioning, AQE, or reduce sh..."
1,High shuffleFetchWaitTime,Tasks waiting for shuffle data,"Check network, skew, remote blocks, executor p..."
2,High diskBytesSpilled,Memory pressure,"Increase memory, reduce partitions, pre-aggreg..."
3,High jvmGCTime,GC pressure,"Tune memory, reduce caching, avoid huge object..."
4,CPU time much lower than run time,Waiting/IO/scheduling overhead,"Look at shuffle wait, GC, spill, input scan, t..."
5,Very high max task time vs median,Skew,"Use partition metrics, AQE skew handling, salt..."
6,Huge resultSize,Driver pressure,Avoid collect/show on large results.


## 14 — Decision checklist

A practical reading order for sparkMeasure reports.

In [15]:
pd.DataFrame([
    (1, "elapsedTime", "Is the job actually slow?"),
    (2, "numStages / numTasks", "Is parallelism reasonable?"),
    (3, "executorCpuTime vs executorRunTime", "CPU-bound or waiting?"),
    (4, "shuffle bytes + fetch wait", "Is shuffle the bottleneck?"),
    (5, "remote shuffle bytes", "Is network involved?"),
    (6, "spill bytes", "Is memory insufficient?"),
    (7, "jvmGCTime", "Is JVM memory pressure visible?"),
    (8, "recordsRead / bytesRead", "How much input was processed?"),
    (9, "resultSize", "Is the driver receiving too much data?"),
], columns=["order", "metric", "question"])

,order,metric,question
0,1,elapsedTime,Is the job actually slow?
1,2,numStages / numTasks,Is parallelism reasonable?
2,3,executorCpuTime vs executorRunTime,CPU-bound or waiting?
3,4,shuffle bytes + fetch wait,Is shuffle the bottleneck?
4,5,remote shuffle bytes,Is network involved?
5,6,spill bytes,Is memory insufficient?
6,7,jvmGCTime,Is JVM memory pressure visible?
7,8,recordsRead / bytesRead,How much input was processed?
8,9,resultSize,Is the driver receiving too much data?


## 15 — Human-readable summary generator

Small helper to convert a sparkMeasure-like metric dictionary into a short diagnosis.

In [16]:
def summarize_metrics(m):
    messages = []
    elapsed = m.get("elapsedTime_ms", 0)
    run = m.get("executorRunTime_ms", 0)
    cpu = m.get("executorCpuTime_ms", 0)
    gc = m.get("jvmGCTime_ms", 0)
    spill = m.get("diskBytesSpilled", 0) + m.get("memoryBytesSpilled", 0)
    shuffle = m.get("shuffleTotalBytesRead", 0)
    remote_shuffle = m.get("shuffleRemoteBytesRead", 0)
    fetch_wait = m.get("shuffleFetchWaitTime_ms", 0)

    messages.append(f"Elapsed time: {elapsed / 1000:.3f}s")
    messages.append(f"CPU efficiency: {cpu / run:.2%}" if run else "CPU efficiency: n/a")
    messages.append(f"Shuffle read: {shuffle / 1024 / 1024:.2f} MB")
    messages.append(f"Remote shuffle read: {remote_shuffle / 1024 / 1024:.2f} MB")
    messages.append(f"Shuffle fetch wait: {fetch_wait} ms")
    messages.append(f"GC time: {gc} ms")
    messages.append(f"Spill: {spill / 1024 / 1024:.2f} MB")

    if spill == 0 and gc < run * 0.05 and fetch_wait == 0 and remote_shuffle == 0:
        messages.append("Diagnosis: healthy small job; no obvious shuffle/network/memory bottleneck.")
    elif spill > 0:
        messages.append("Diagnosis: memory pressure / spilling.")
    elif fetch_wait > 0 or remote_shuffle > 0:
        messages.append("Diagnosis: shuffle/network overhead.")
    elif run and cpu / run < 0.5:
        messages.append("Diagnosis: tasks spend significant time waiting.")
    else:
        messages.append("Diagnosis: inspect plan and task distribution.")

    return "\n".join(messages)

print(summarize_metrics(metrics))

Elapsed time: 0.700s
CPU efficiency: 98.61%
Shuffle read: 3.84 MB
Remote shuffle read: 0.00 MB
Shuffle fetch wait: 0 ms
GC time: 10 ms
Spill: 0.00 MB
Diagnosis: healthy small job; no obvious shuffle/network/memory bottleneck.


## 16 — Final interpretation

For this exact report: the job is healthy. Spark performed a small local shuffle, but there is no spill, no remote shuffle, almost no GC, and high CPU efficiency.

In [17]:
pd.DataFrame([
    ("Do I need to optimize this job?", "Probably not", "The report is already healthy."),
    ("Is shuffle a problem?", "No", "Only 3.8 MB and fully local."),
    ("Is memory a problem?", "No", "No spill and tiny GC."),
    ("Is network a problem?", "No", "0 remote shuffle bytes."),
    ("Is CPU being used?", "Yes", "executorCpuTime is close to executorRunTime."),
    ("What should I watch next?", "Task distribution", "If future jobs are slow, inspect skew/max task duration."),
], columns=["question", "answer", "reason"])

,question,answer,reason
0,Do I need to optimize this job?,Probably not,The report is already healthy.
1,Is shuffle a problem?,No,Only 3.8 MB and fully local.
2,Is memory a problem?,No,No spill and tiny GC.
3,Is network a problem?,No,0 remote shuffle bytes.
4,Is CPU being used?,Yes,executorCpuTime is close to executorRunTime.
5,What should I watch next?,Task distribution,"If future jobs are slow, inspect skew/max task..."
